# 05 — Final Model Training & Ensemble

This notebook makes the **final modeling decision** for the Moneyball season-wins project.

By this point:

- `02_feature_engineering.ipynb` created the canonical model-ready features;
- `03_model_feature_selection.ipynb` selected each model's best feature set;
- `04_cluster_feature_experiment.ipynb` showed that KMeans features do not add enough value to keep in the main pipeline.

The remaining question is:

> **Should the final prediction use one strong standalone model, or can a linear + nonlinear ensemble reduce MAE further?**

### Objectives

1. Reuse each candidate model's **best feature set from 03**.
2. Generate leakage-safe **out-of-fold (OOF) predictions** for ElasticNet, Ridge, HGB, and LightGBM.
3. Compare standalone MAE and **residual correlation** to see whether the models make different mistakes.
4. Test a small set of weighted ensembles, especially **linear + nonlinear** combinations.
5. Run a **secondary chronological holdout** as a sanity check.
6. Select the final standalone model or ensemble, fit it on all training rows, and create the submission.

### Important decision

LightGBM is kept only as an **ensemble challenger**. It was weaker as a standalone model in 03, but a weaker model can still help if its residuals are sufficiently different.

### Outputs

Written to `outputs/final_model/`:

- `05_oof_predictions.csv`
- `05_model_results.csv`
- `05_residual_correlations.csv`
- `05_chronological_results.csv`
- `05_ensemble_results.csv`
- `05_final_decision.csv`

Final submission:

- `outputs/submissions/submission_predict.csv`


## 1. Setup and load inputs

05 does not rebuild features and does not redo feature selection. It consumes the decisions made upstream.


In [ ]:
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold

from moneyball import project_config as cfg
from moneyball import feature_sets as fs
from moneyball import model_factory as mf

cfg.configure_notebook()
cfg.ensure_project_dirs()

TRAIN_FE_PATH        = cfg.TRAIN_FE_PATH
PRED_FE_PATH         = cfg.PRED_FE_PATH
RESULTS_03_PATH      = cfg.MODEL_RESULT_PATH
CLUSTER_SUMMARY_PATH = cfg.CLUSTER_SUMMARY_PATH

FINAL_DIR = cfg.OUTPUT_DIR / "final_model"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_DIR = cfg.OUTPUT_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

OOF_PATH = FINAL_DIR / "05a_oof_predictions.csv"
MODEL_RESULT_PATH = FINAL_DIR / "05a_model_results.csv"
RESIDUAL_CORR_PATH = FINAL_DIR / "05a_residual_correlations.csv"
CHRONO_PATH = FINAL_DIR / "05a_chronological_results.csv"
ENSEMBLE_PATH = FINAL_DIR / "05a_ensemble_results.csv"
DECISION_PATH = FINAL_DIR / "05a_final_decision.csv"
SUBMISSION_PATH = SUBMISSION_DIR / "submission_5a.csv"

print("Train FE     :", TRAIN_FE_PATH)
print("Prediction FE:", PRED_FE_PATH)
print("03 results   :", RESULTS_03_PATH)
print("Final outputs:", FINAL_DIR)

Train FE     : /home/shpang/devs/ntu/projects/baseball_v2/data/processed/train_fe.csv
Prediction FE: /home/shpang/devs/ntu/projects/baseball_v2/data/processed/pred_fe.csv
03 results   : /home/shpang/devs/ntu/projects/baseball_v2/outputs/model_selection/03_model_feature_results.csv
Final outputs: /home/shpang/devs/ntu/projects/baseball_v2/outputs/final_model


In [18]:
train_fe = pd.read_csv(TRAIN_FE_PATH)
pred_fe = pd.read_csv(PRED_FE_PATH)

if not RESULTS_03_PATH.exists():
    raise FileNotFoundError(
        "03_model_feature_results.csv was not found. Run 03 first."
    )

results_03 = pd.read_csv(RESULTS_03_PATH)

FEATURE_SETS = fs.get_candidate_feature_sets(train_fe)
fs.validate_feature_sets(train_fe, pred_fe, FEATURE_SETS)

print("Train shape:", train_fe.shape)
print("Pred shape :", pred_fe.shape)

if CLUSTER_SUMMARY_PATH.exists():
    print("\n04 cluster summary (reference only):")
    display(pd.read_csv(CLUSTER_SUMMARY_PATH))

Train shape: (1812, 78)
Pred shape : (453, 76)

04 cluster summary (reference only):


,model,feature_set,variant,n_clusters,baseline_mae,cluster_mae,delta_mae,cluster_mae_std,folds_improved,best_variant,best_k,best_mae,best_mae_std,total_n_features
0,elasticnet,all_domain_raw,cluster_k8,8.0,2.727792,2.734371,0.006579,0.092397,2,baseline,8.0,2.727792,0.089080,72
1,hgb,all_domain_raw,cluster_k4,4.0,3.070202,3.064434,-0.005768,0.059647,4,cluster_k4,4.0,3.064434,0.059647,76
2,ridge,all_domain_raw,cluster_k8,8.0,2.729300,2.737869,0.008569,0.085010,1,baseline,8.0,2.729300,0.101873,72


## 2. Reconstruct each candidate's best configuration from 03

We carry four models into the final analysis:

- **ElasticNet** — strongest linear result;
- **Ridge** — essentially tied with ElasticNet;
- **HGB** — strongest nonlinear model;
- **LightGBM** — weaker standalone, kept only to test ensemble diversity.

Each model keeps the feature set on which it actually performed best in 03.


In [19]:
CANDIDATE_MODELS = ["elasticnet", "ridge", "hgb", "lightgbm"]

missing = sorted(set(CANDIDATE_MODELS) - set(results_03["model"]))
if missing:
    raise ValueError(f"03 results are missing candidate models: {missing}")

best_config = (
    results_03[results_03["model"].isin(CANDIDATE_MODELS)]
    .sort_values(["mae_mean", "mae_std"])
    .groupby("model", as_index=False)
    .first()
    .set_index("model")
    .loc[CANDIDATE_MODELS]
    .reset_index()
)

unknown_sets = sorted(set(best_config["feature_set"]) - set(FEATURE_SETS))
if unknown_sets:
    raise ValueError(f"Unknown feature sets from 03: {unknown_sets}")

display(best_config[
    ["model", "feature_set", "n_features", "mae_mean", "mae_std"]
].round(4))

,model,feature_set,n_features,mae_mean,mae_std
0,elasticnet,all_domain_raw,72,2.7278,0.0891
1,ridge,all_domain_raw,72,2.7293,0.1019
2,hgb,all_domain_raw,72,3.0702,0.0657
3,lightgbm,core_pitch,18,3.1159,0.0517


## 3. Primary validation — OOF predictions

We keep the same primary validation design used upstream:

- 5-fold `GroupKFold`
- grouped by `meta_yearID`
- target = `target_W`
- metric = MAE in wins

The difference now is that we save **one prediction for every training row**. Those OOF predictions let us compare residuals and test ensembles fairly.


In [20]:
TARGET_COL = "target_W"
GROUP_COL = "meta_yearID"
N_SPLITS = 5

y = train_fe[TARGET_COL].astype(float)
groups = train_fe[GROUP_COL]
cv = GroupKFold(n_splits=N_SPLITS)

fold_assignment = np.zeros(len(train_fe), dtype=int)
for fold_no, (_, valid_idx) in enumerate(cv.split(train_fe, y, groups=groups), start=1):
    fold_assignment[valid_idx] = fold_no

assert (fold_assignment > 0).all()

In [21]:
def generate_oof(model_name, feature_set_name):
    """Generate one leakage-safe OOF prediction for every training row."""
    features = FEATURE_SETS[feature_set_name]
    X = train_fe[features]
    pred = np.full(len(train_fe), np.nan)

    with warnings.catch_warnings():
        warnings.simplefilter("once", ConvergenceWarning)

        for train_idx, valid_idx in cv.split(X, y, groups=groups):
            model = mf.MODEL_FACTORIES[model_name]()
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            pred[valid_idx] = model.predict(X.iloc[valid_idx])

    if np.isnan(pred).any():
        raise RuntimeError(f"Incomplete OOF predictions for {model_name}")

    return pred


oof = pd.DataFrame({
    "meta_ID": train_fe["meta_ID"],
    "meta_yearID": train_fe["meta_yearID"],
    "fold": fold_assignment,
    "actual_W": y,
})

for _, row in best_config.iterrows():
    model_name = row["model"]
    print(f"OOF: {model_name} + {row['feature_set']}")
    oof[f"pred_{model_name}"] = generate_oof(
        model_name,
        row["feature_set"],
    )

oof.to_csv(OOF_PATH, index=False)
display(oof.head())

OOF: elasticnet + all_domain_raw


OOF: ridge + all_domain_raw
OOF: hgb + all_domain_raw
OOF: lightgbm + core_pitch


/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, 

,meta_ID,meta_yearID,fold,actual_W,pred_elasticnet,pred_ridge,pred_hgb,pred_lightgbm
0,317,1935,4,78.0,77.143541,77.368252,76.092919,75.747229
1,2162,1993,5,86.0,89.314393,89.810519,90.534685,90.244798
2,1895,2016,5,86.0,88.652666,88.678974,90.251393,90.156696
3,428,1938,5,89.0,89.406069,89.703694,88.434008,90.764088
4,375,1996,3,85.0,81.157340,81.264319,83.778098,82.843575


## 4. Standalone results and residual correlation

Residuals are:

`actual wins - OOF prediction`

A weaker nonlinear model can still be useful in an ensemble if it makes **different mistakes** from the strongest linear model.


In [22]:
model_rows = []
residuals = pd.DataFrame(index=oof.index)

for _, row in best_config.iterrows():
    model_name = row["model"]
    pred_col = f"pred_{model_name}"

    fold_mae = []
    for fold_no in range(1, N_SPLITS + 1):
        f = oof[oof["fold"] == fold_no]
        fold_mae.append(mean_absolute_error(f["actual_W"], f[pred_col]))

    model_rows.append({
        "model": model_name,
        "feature_set": row["feature_set"],
        "n_features": int(row["n_features"]),
        "oof_mae": mean_absolute_error(oof["actual_W"], oof[pred_col]),
        "fold_mae_mean": float(np.mean(fold_mae)),
        "fold_mae_std": float(np.std(fold_mae)),
    })

    residuals[model_name] = oof["actual_W"] - oof[pred_col]

model_results = pd.DataFrame(model_rows).sort_values("oof_mae").reset_index(drop=True)
residual_corr = residuals.corr()

model_results.to_csv(MODEL_RESULT_PATH, index=False)
residual_corr.to_csv(RESIDUAL_CORR_PATH)

print("Standalone OOF results")
display(model_results.round(4))

print("Residual correlation")
display(residual_corr.round(3))

Standalone OOF results


,model,feature_set,n_features,oof_mae,fold_mae_mean,fold_mae_std
0,elasticnet,all_domain_raw,72,2.7275,2.7278,0.0891
1,ridge,all_domain_raw,72,2.7290,2.7293,0.1019
2,hgb,all_domain_raw,72,3.0703,3.0702,0.0657
3,lightgbm,core_pitch,18,3.1160,3.1159,0.0517


Residual correlation


,elasticnet,ridge,hgb,lightgbm
elasticnet,1.000,0.994,0.873,0.866
ridge,0.994,1.000,0.854,0.846
hgb,0.873,0.854,1.000,0.936
lightgbm,0.866,0.846,0.936,1.000


### How to interpret residual correlation

- Ridge and ElasticNet are expected to be highly correlated because both are regularized linear models.
- HGB and LightGBM only become useful ensemble partners if their residuals are sufficiently different.
- Correlation alone is not enough: the blend must still improve MAE.


## 5. Controlled ensemble test

The better of Ridge and ElasticNet becomes the **linear anchor**.

We use two different search ranges:

- **Ridge / ElasticNet:** search broadly because the earlier run showed MAE was still improving at the old 30% boundary.
- **HGB / LightGBM:** keep partner weights small because both nonlinear models are materially weaker standalone.

### Weight ranges

**Other linear model**

- 5% to 90% partner weight, in 5% steps.

**HGB / LightGBM**

- 5%, 10%, 15%, 20%, 25%, 30%.

At this point we calculate **float OOF MAE** for every blend.

We do **not** make the final choice yet. After the chronological predictions are available, we will score every configuration in four ways:

- float OOF MAE;
- rounded OOF MAE;
- float chronological MAE;
- rounded chronological MAE.

That prevents us from choosing ensemble weights using float predictions and only later discovering that a different blend is better once predictions are rounded to whole wins.


In [23]:
linear_anchor = (
    model_results[model_results["model"].isin(["elasticnet", "ridge"])]
    .sort_values("oof_mae")
    .iloc[0]["model"]
)

partners = [m for m in CANDIDATE_MODELS if m != linear_anchor]

# Search the other linear model much more broadly.
# The earlier run was still improving at the old 30% boundary, so we extend
# the grid instead of assuming 70/30 was optimal.
LINEAR_PARTNER_WEIGHTS = [
    0.05, 0.10, 0.15, 0.20, 0.25, 0.30,
    0.35, 0.40, 0.45, 0.50, 0.55, 0.60,
    0.65, 0.70, 0.75, 0.80, 0.85, 0.90,
]

# Keep nonlinear partner weights small because HGB and LightGBM were
# substantially weaker standalone and only showed value, if any, as a
# small diversity correction.
NONLINEAR_PARTNER_WEIGHTS = [
    0.05, 0.10, 0.15, 0.20, 0.25, 0.30,
]

print("Linear anchor:", linear_anchor)
print("Partners     :", partners)
print("Linear grid  :", LINEAR_PARTNER_WEIGHTS)
print("Nonlinear grid:", NONLINEAR_PARTNER_WEIGHTS)

Linear anchor: elasticnet
Partners     : ['ridge', 'hgb', 'lightgbm']
Linear grid  : [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
Nonlinear grid: [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]


In [24]:
# ---------------------------------------------------------------------
# Build the ensemble grid and calculate FLOAT OOF MAE
# ---------------------------------------------------------------------
#
# This creates `ensemble_results`, which is used later when we add
# chronological and rounded scores for every configuration.

ensemble_rows = []

anchor_pred = oof[f"pred_{linear_anchor}"].to_numpy()
actual = oof["actual_W"].to_numpy()

# Include the standalone linear anchor.
ensemble_rows.append(
    {
        "configuration": linear_anchor,
        "anchor_model": linear_anchor,
        "partner_model": None,
        "anchor_weight": 1.0,
        "partner_weight": 0.0,
        "float_oof_mae": mean_absolute_error(
            actual,
            anchor_pred,
        ),
    }
)

for partner in partners:
    partner_pred = oof[f"pred_{partner}"].to_numpy()

    # Broader search for the other linear model.
    if partner in {"elasticnet", "ridge"}:
        partner_weights = LINEAR_PARTNER_WEIGHTS
    else:
        partner_weights = NONLINEAR_PARTNER_WEIGHTS

    for partner_weight in partner_weights:
        anchor_weight = 1.0 - partner_weight

        blended_pred = (
            anchor_weight * anchor_pred
            + partner_weight * partner_pred
        )

        ensemble_rows.append(
            {
                "configuration": (
                    f"{linear_anchor}_{anchor_weight:.2f}"
                    f"__{partner}_{partner_weight:.2f}"
                ),
                "anchor_model": linear_anchor,
                "partner_model": partner,
                "anchor_weight": anchor_weight,
                "partner_weight": partner_weight,
                "float_oof_mae": mean_absolute_error(
                    actual,
                    blended_pred,
                ),
            }
        )

ensemble_results = (
    pd.DataFrame(ensemble_rows)
    .sort_values("float_oof_mae")
    .reset_index(drop=True)
)

print("Float OOF ensemble grid")
display(ensemble_results.head(20).round(4))


Float OOF ensemble grid


,configuration,anchor_model,partner_model,anchor_weight,partner_weight,float_oof_mae
0,elasticnet_0.55__ridge_0.45,elasticnet,ridge,0.55,0.45,2.7244
1,elasticnet_0.50__ridge_0.50,elasticnet,ridge,0.50,0.50,2.7244
2,elasticnet_0.60__ridge_0.40,elasticnet,ridge,0.60,0.40,2.7244
3,elasticnet_0.45__ridge_0.55,elasticnet,ridge,0.45,0.55,2.7245
4,elasticnet_0.65__ridge_0.35,elasticnet,ridge,0.65,0.35,2.7245
5,elasticnet_0.40__ridge_0.60,elasticnet,ridge,0.40,0.60,2.7247
6,elasticnet_0.70__ridge_0.30,elasticnet,ridge,0.70,0.30,2.7247
7,elasticnet_0.75__ridge_0.25,elasticnet,ridge,0.75,0.25,2.7249
8,elasticnet_0.35__ridge_0.65,elasticnet,ridge,0.35,0.65,2.7250
9,elasticnet_0.80__ridge_0.20,elasticnet,ridge,0.80,0.20,2.7253


## 6. Secondary chronological holdout

GroupKFold remains the primary comparison because it is consistent with 03 and 04.

As a final sanity check, we also train on the earliest 80% of unique seasons and validate on the latest 20%.

This asks a different question:

> Does the selected model still behave sensibly when predicting later historical seasons?


In [25]:
years = np.array(sorted(train_fe[GROUP_COL].unique()))
split_at = int(len(years) * 0.80)

train_years = years[:split_at]
valid_years = years[split_at:]

train_mask = train_fe[GROUP_COL].isin(train_years)
valid_mask = train_fe[GROUP_COL].isin(valid_years)

chrono_actual = y.loc[valid_mask].to_numpy()
chrono_pred = {}

config_lookup = best_config.set_index("model").to_dict("index")

for model_name in CANDIDATE_MODELS:
    feature_set_name = config_lookup[model_name]["feature_set"]
    features = FEATURE_SETS[feature_set_name]

    model = mf.MODEL_FACTORIES[model_name]()
    model.fit(train_fe.loc[train_mask, features], y.loc[train_mask])
    chrono_pred[model_name] = model.predict(train_fe.loc[valid_mask, features])

print(
    f"Chronological train: {train_years.min()}–{train_years.max()} "
    f"({train_mask.sum()} rows)"
)
print(
    f"Chronological valid: {valid_years.min()}–{valid_years.max()} "
    f"({valid_mask.sum()} rows)"
)

Chronological train: 1904–1992 (1275 rows)
Chronological valid: 1993–2016 (537 rows)


/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [26]:
# Score every ensemble configuration in BOTH prediction forms:
#   1) continuous (float) wins
#   2) rounded whole wins
#
# We do this for both validation views before choosing the final weights.

scored_rows = []

for _, row in ensemble_results.iterrows():
    anchor = row["anchor_model"]
    partner = row["partner_model"]

    # -------------------------------------------------------------
    # OOF prediction for this exact blend
    # -------------------------------------------------------------
    oof_prediction = (
        row["anchor_weight"]
        * oof[f"pred_{anchor}"].to_numpy()
    )

    if not pd.isna(partner):
        oof_prediction += (
            row["partner_weight"]
            * oof[f"pred_{partner}"].to_numpy()
        )

    # -------------------------------------------------------------
    # Chronological prediction for the same exact blend
    # -------------------------------------------------------------
    chrono_prediction = (
        row["anchor_weight"]
        * chrono_pred[anchor]
    )

    if not pd.isna(partner):
        chrono_prediction += (
            row["partner_weight"]
            * chrono_pred[partner]
        )

    scored_rows.append(
        {
            **row.to_dict(),
            "rounded_oof_mae": mean_absolute_error(
                actual,
                np.rint(oof_prediction),
            ),
            "float_chrono_mae": mean_absolute_error(
                chrono_actual,
                chrono_prediction,
            ),
            "rounded_chrono_mae": mean_absolute_error(
                chrono_actual,
                np.rint(chrono_prediction),
            ),
        }
    )

ensemble_results = pd.DataFrame(scored_rows)

# Keep the float-OFF ordering visible first; the final decision section
# will explicitly compare float and rounded winners.
ensemble_results = ensemble_results.sort_values(
    ["float_oof_mae", "rounded_oof_mae"]
).reset_index(drop=True)

ensemble_results.to_csv(
    ENSEMBLE_PATH,
    index=False,
)

# Save the standalone chronological view as a separate reference table.
chrono_rows = []
for model_name in CANDIDATE_MODELS:
    chrono_rows.append(
        {
            "configuration": model_name,
            "float_chrono_mae": mean_absolute_error(
                chrono_actual,
                chrono_pred[model_name],
            ),
            "rounded_chrono_mae": mean_absolute_error(
                chrono_actual,
                np.rint(chrono_pred[model_name]),
            ),
        }
    )

chrono_results = pd.DataFrame(chrono_rows)
chrono_results.to_csv(
    CHRONO_PATH,
    index=False,
)

print(f"Saved ensemble results to: {ENSEMBLE_PATH}")
print(f"Saved chronological standalone results to: {CHRONO_PATH}")

display(
    ensemble_results[
        [
            "configuration",
            "float_oof_mae",
            "rounded_oof_mae",
            "float_chrono_mae",
            "rounded_chrono_mae",
            "anchor_model",
            "partner_model",
            "partner_weight",
        ]
    ]
    .head(20)
    .round(4)
)

Saved ensemble results to: /home/shpang/devs/ntu/projects/baseball_v2/outputs/final_model/05_ensemble_results.csv
Saved chronological standalone results to: /home/shpang/devs/ntu/projects/baseball_v2/outputs/final_model/05_chronological_results.csv


,configuration,float_oof_mae,rounded_oof_mae,float_chrono_mae,rounded_chrono_mae,anchor_model,partner_model,partner_weight
0,elasticnet_0.55__ridge_0.45,2.7244,2.7081,2.3687,2.3557,elasticnet,ridge,0.45
1,elasticnet_0.50__ridge_0.50,2.7244,2.7097,2.3678,2.3613,elasticnet,ridge,0.50
2,elasticnet_0.60__ridge_0.40,2.7244,2.7070,2.3697,2.3464,elasticnet,ridge,0.40
3,elasticnet_0.45__ridge_0.55,2.7245,2.7136,2.3668,2.3669,elasticnet,ridge,0.55
4,elasticnet_0.65__ridge_0.35,2.7245,2.7092,2.3707,2.3501,elasticnet,ridge,0.35
5,elasticnet_0.40__ridge_0.60,2.7247,2.7103,2.3659,2.3724,elasticnet,ridge,0.60
6,elasticnet_0.70__ridge_0.30,2.7247,2.7075,2.3718,2.3501,elasticnet,ridge,0.30
7,elasticnet_0.75__ridge_0.25,2.7249,2.7064,2.3729,2.3575,elasticnet,ridge,0.25
8,elasticnet_0.35__ridge_0.65,2.7250,2.7108,2.3651,2.3724,elasticnet,ridge,0.65
9,elasticnet_0.80__ridge_0.20,2.7253,2.7075,2.3741,2.3575,elasticnet,ridge,0.20


## 7. Final decision

We now have four validation scores for **every** standalone / ensemble configuration:

- float OOF MAE;
- rounded OOF MAE;
- float chronological MAE;
- rounded chronological MAE.

### Decision rule

1. **OOF MAE remains the primary ranking metric.**
2. Compare the best float configuration against the best rounded configuration.
3. If rounding gives the lower OOF MAE **and** its chronological MAE is not worse than the corresponding float result, use rounded predictions.
4. If two configurations are effectively tied, prefer the simpler / more balanced blend rather than chasing a tiny weight difference.
5. The chronological holdout is a secondary robustness check, not the optimization target.

This means the final ensemble weights and the decision to round are selected **together**, rather than in two separate steps.


In [27]:
# -------------------------------------------------------------
# Find the best FLOAT configuration
# -------------------------------------------------------------
best_float = (
    ensemble_results
    .sort_values(["float_oof_mae", "float_chrono_mae"])
    .iloc[0]
)

# -------------------------------------------------------------
# Find the best ROUNDED configuration
# -------------------------------------------------------------
best_rounded = (
    ensemble_results
    .sort_values(["rounded_oof_mae", "rounded_chrono_mae"])
    .iloc[0]
)

print("Best float configuration")
display(
    best_float[
        [
            "configuration",
            "float_oof_mae",
            "float_chrono_mae",
            "anchor_model",
            "partner_model",
            "anchor_weight",
            "partner_weight",
        ]
    ].to_frame("value")
)

print("\nBest rounded configuration")
display(
    best_rounded[
        [
            "configuration",
            "rounded_oof_mae",
            "rounded_chrono_mae",
            "anchor_model",
            "partner_model",
            "anchor_weight",
            "partner_weight",
        ]
    ].to_frame("value")
)

# -------------------------------------------------------------
# Automatic recommendation
# -------------------------------------------------------------
# Rounding is selected only when it improves the primary OOF metric and
# remains at least as good on the chronological sanity check.
use_rounded = (
    best_rounded["rounded_oof_mae"]
    < best_float["float_oof_mae"]
    and best_rounded["rounded_chrono_mae"]
    <= best_float["float_chrono_mae"]
)

AUTO_BEST_CONFIG = (
    best_rounded["configuration"]
    if use_rounded
    else best_float["configuration"]
)

AUTO_PREDICTION_FORM = (
    "rounded"
    if use_rounded
    else "float"
)

# Human-in-the-loop final choice.
#
# Change these only if the validation tables above give a clear reason
# to override the automatic recommendation.
FINAL_CONFIG = AUTO_BEST_CONFIG
FINAL_PREDICTION_FORM = AUTO_PREDICTION_FORM

selected = ensemble_results[
    ensemble_results["configuration"] == FINAL_CONFIG
].iloc[0]

USE_ROUNDED_PREDICTIONS = (
    FINAL_PREDICTION_FORM == "rounded"
)

print("\nAutomatic configuration :", AUTO_BEST_CONFIG)
print("Automatic prediction form:", AUTO_PREDICTION_FORM)
print("Selected configuration   :", FINAL_CONFIG)
print("Selected prediction form :", FINAL_PREDICTION_FORM)

Best float configuration


,value
configuration,elasticnet_0.55__ridge_0.45
float_oof_mae,2.724362
float_chrono_mae,2.368721
anchor_model,elasticnet
partner_model,ridge
anchor_weight,0.55
partner_weight,0.45



Best rounded configuration


,value
configuration,elasticnet_0.90__ridge_0.10
rounded_oof_mae,2.703091
rounded_chrono_mae,2.361266
anchor_model,elasticnet
partner_model,ridge
anchor_weight,0.9
partner_weight,0.1



Automatic configuration : elasticnet_0.90__ridge_0.10
Automatic prediction form: rounded
Selected configuration   : elasticnet_0.90__ridge_0.10
Selected prediction form : rounded


## 8. Confirm the selected validation scores

Float-vs-rounded scoring has already been performed across **every ensemble configuration**.

This section simply records the four metrics for the selected final configuration so they are easy to read and save with the decision record.


In [28]:
float_oof_mae = selected["float_oof_mae"]
rounded_oof_mae = selected["rounded_oof_mae"]
float_chrono_mae = selected["float_chrono_mae"]
rounded_chrono_mae = selected["rounded_chrono_mae"]

print(f"Float OOF MAE       : {float_oof_mae:.4f}")
print(f"Rounded OOF MAE     : {rounded_oof_mae:.4f}")
print(f"Float Chrono MAE    : {float_chrono_mae:.4f}")
print(f"Rounded Chrono MAE  : {rounded_chrono_mae:.4f}")
print("Submission form    :", FINAL_PREDICTION_FORM)

Float OOF MAE       : 2.7263
Rounded OOF MAE     : 2.7031
Float Chrono MAE    : 2.3766
Rounded Chrono MAE  : 2.3613
Submission form    : rounded


## 9. Fit the selected configuration on all training rows

Only the model(s) with non-zero final weight are fitted.

Each model keeps its own 03-selected feature set. This matters especially if LightGBM survives as an ensemble partner.


In [29]:
def fit_full_predict(model_name):
    config = config_lookup[model_name]
    features = FEATURE_SETS[config["feature_set"]]

    model = mf.MODEL_FACTORIES[model_name]()
    model.fit(train_fe[features], y)

    return model, model.predict(pred_fe[features])


final_models = {}
final_prediction = np.zeros(len(pred_fe), dtype=float)

anchor_model, anchor_pred = fit_full_predict(selected["anchor_model"])
final_models[selected["anchor_model"]] = anchor_model
final_prediction += selected["anchor_weight"] * anchor_pred

if not pd.isna(selected["partner_model"]):
    partner_model, partner_pred = fit_full_predict(selected["partner_model"])
    final_models[selected["partner_model"]] = partner_model
    final_prediction += selected["partner_weight"] * partner_pred

# Season wins cannot be below 0 or above games played.
final_prediction = np.clip(
    final_prediction,
    0,
    pred_fe["season_G"].to_numpy(),
)

if USE_ROUNDED_PREDICTIONS:
    final_prediction = np.rint(final_prediction)

print("Fitted models:", list(final_models))
print(
    "Prediction range:",
    float(final_prediction.min()),
    "to",
    float(final_prediction.max()),
)

Fitted models: ['elasticnet', 'ridge']
Prediction range: 44.0 to 109.0


## 10. Create the submission and save the decision record

The decision record captures the exact models, feature sets, weights, and prediction form used so the final submission can be reproduced later.


In [30]:
submission = pd.DataFrame({
    cfg.ID_COL: pred_fe["meta_ID"],
    cfg.TARGET_COL: final_prediction,
})

submission.to_csv(SUBMISSION_PATH, index=False)

decision = pd.DataFrame([{
    "final_configuration": FINAL_CONFIG,
    "float_oof_mae": float_oof_mae,
    "rounded_oof_mae": rounded_oof_mae,
    "float_chronological_mae": float_chrono_mae,
    "rounded_chronological_mae": rounded_chrono_mae,
    "anchor_model": selected["anchor_model"],
    "anchor_feature_set": config_lookup[selected["anchor_model"]]["feature_set"],
    "anchor_weight": selected["anchor_weight"],
    "partner_model": selected["partner_model"],
    "partner_feature_set": (
        None
        if pd.isna(selected["partner_model"])
        else config_lookup[selected["partner_model"]]["feature_set"]
    ),
    "partner_weight": selected["partner_weight"],
    "prediction_form": FINAL_PREDICTION_FORM,
    "rounded_predictions": USE_ROUNDED_PREDICTIONS,
}])

decision.to_csv(DECISION_PATH, index=False)

print("Saved submission     :", SUBMISSION_PATH)
print("Saved decision record:", DECISION_PATH)

display(submission.head())
display(decision.T)

Saved submission     : /home/shpang/devs/ntu/projects/baseball_v2/outputs/submissions/submission_predict.csv
Saved decision record: /home/shpang/devs/ntu/projects/baseball_v2/outputs/final_model/05_final_decision.csv


,ID,W
0,1756,69.0
1,1282,74.0
2,351,84.0
3,421,87.0
4,57,93.0


,0
final_configuration,elasticnet_0.90__ridge_0.10
float_oof_mae,2.726281
rounded_oof_mae,2.703091
float_chronological_mae,2.376578
rounded_chronological_mae,2.361266
anchor_model,elasticnet
anchor_feature_set,all_domain_raw
anchor_weight,0.9
partner_model,ridge
partner_feature_set,all_domain_raw


## 11. Final interpretation

After running the notebook, record the actual conclusion here in plain language.

### Observation

The strongest continuous ensemble was a near-balanced ElasticNet/Ridge blend, but after evaluating all candidate blends using whole-win predictions, the lowest OOF MAE was achieved by 90% ElasticNet + 10% Ridge. Rounding reduced OOF MAE from the best continuous result of 2.7244 to 2.7031, and the selected rounded ensemble also remained strong on the chronological holdout at 2.3613 MAE.

### Decision

Use a 90% ElasticNet + 10% Ridge ensemble with predictions rounded to the nearest whole win for the final submission.

### Reasoning

ElasticNet and Ridge were the two strongest standalone models throughout the cleaned pipeline. Nonlinear HGB and LightGBM models were tested as ensemble partners but did not improve validation enough to justify their additional complexity. The selected linear ensemble produced the best rounded OOF result in the tested grid, while also remaining robust on the later-season chronological check.